In [13]:
import sys,os
import pandas as pd
import numpy as np

# Add src directory to path
sys.path.append("../src")

# Import model helper functions
from model import prepare_features, split_data, train_logistic_regression, train_random_forest, evaluate_model

In [14]:
# Load processed dataset
data_path = "../data/processed/bitcoin_processed.csv"
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Could not find processed data at {data_path}.")

df = pd.read_csv(data_path)
print(f"Loaded dataset with {len(df)} rows and {len(df.columns)} columns.")
df.head()

Loaded dataset with 365 rows and 24 columns.


,date,price,volume,market_cap,fear_greed_value,fear_greed_label,daily_sentiment_score,negative_headline_count,headline_count,daily_return,...,next_day_return,volatility_scaled,volume_spike_scaled,sentiment_negativity_scaled,market_stress_index,fear_component,price_drop_component,price_drop_component_scaled,panic_score,is_panic_day
0,2025-06-13,105979.229024,3.586516e+10,2.107912e+12,61,Greed,NaN,NaN,NaN,NaN,...,0.000626,NaN,NaN,NaN,NaN,0.39,0.000000,0.000000,NaN,0
1,2025-06-14,106045.564408,4.947102e+10,2.107973e+12,63,Greed,NaN,NaN,NaN,0.000626,...,-0.005306,NaN,NaN,NaN,NaN,0.37,0.000000,0.000000,NaN,0
2,2025-06-15,105482.906116,1.791641e+10,2.097487e+12,60,Greed,NaN,NaN,NaN,-0.005306,...,0.000679,NaN,NaN,NaN,NaN,0.40,0.005306,0.115673,NaN,0
3,2025-06-16,105554.493831,1.656844e+10,2.098499e+12,61,Greed,NaN,NaN,NaN,0.000679,...,0.013233,NaN,NaN,NaN,NaN,0.39,0.000000,0.000000,NaN,0
4,2025-06-17,106951.272018,3.209596e+10,2.127895e+12,68,Greed,NaN,NaN,NaN,0.013233,...,-0.021204,NaN,NaN,NaN,NaN,0.32,0.000000,0.000000,NaN,0


In [15]:
# Prepare features and target variable
X, y = prepare_features(df)
print(f"Prepared features shape: {X.shape}, target shape: {y.shape}")
print("\nTarget (1 = Next day return is negative, 0 = Otherwise) distribution:")
print(y.value_counts(normalize=True))

Prepared features shape: (25, 7), target shape: (25,)

Target (1 = Next day return is negative, 0 = Otherwise) distribution:
next_day_down
1    0.8
0    0.2
Name: proportion, dtype: float64


In [16]:
# Split data sequentially (no shuffling) - 80% train, 20% test
X_train, X_test, y_train, y_test = split_data(X, y)
print(f"Train set size: {X_train.shape[0]} rows (from index {X_train.index[0]} to {X_train.index[-1]})")
print(f"Test set size: {X_test.shape[0]} rows (from index {X_test.index[0]} to {X_test.index[-1]})")
print(f"\nTrain set target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\nTest set target distribution:\n{y_test.value_counts(normalize=True)}")

Train set size: 20 rows (from index 333 to 358)
Test set size: 5 rows (from index 359 to 363)

Train set target distribution:
next_day_down
1    0.85
0    0.15
Name: proportion, dtype: float64

Test set target distribution:
next_day_down
1    0.6
0    0.4
Name: proportion, dtype: float64


In [17]:
# Train Logistic Regression and Random Forest models
print("Training Logistic Regression model...")
lr_model = train_logistic_regression(X_train, y_train)

print("Training Random Forest classifier...")
rf_model = train_random_forest(X_train, y_train)

Training Logistic Regression model...
Training Random Forest classifier...


In [18]:
# Evaluate models
lr_results = evaluate_model(lr_model, X_test, y_test, "Logistic Regression")
rf_results = evaluate_model(rf_model, X_test, y_test, "Random Forest")

print("\n=== LOGISTIC REGRESSION EVALUATION ===")
for k, v in lr_results.items():
    if k != "confusion_matrix":
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")
    else:
        print(f"{k}:\n{np.array(v)}")

print("\n=== RANDOM FOREST EVALUATION ===")
for k, v in rf_results.items():
    if k != "confusion_matrix":
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")
    else:
        print(f"{k}:\n{np.array(v)}")


=== LOGISTIC REGRESSION EVALUATION ===
model_name: Logistic Regression
accuracy: 0.6000
precision: 0.6000
recall: 1.0000
f1_score: 0.7500
confusion_matrix:
[[0 2]
 [0 3]]

=== RANDOM FOREST EVALUATION ===
model_name: Random Forest
accuracy: 0.6000
precision: 0.6000
recall: 1.0000
f1_score: 0.7500
confusion_matrix:
[[0 2]
 [0 3]]


In [19]:
# Save comparison results to outputs/model_results.csv
results_df = pd.DataFrame([lr_results, rf_results])
# Drop confusion matrix column for flat CSV reporting
results_csv_df = results_df.drop(columns=["confusion_matrix"])

os.makedirs("../outputs", exist_ok=True)
results_csv_path = "../outputs/model_results.csv"
results_csv_df.to_csv(results_csv_path, index=False)
print(f"Successfully saved model results comparison to: {results_csv_path}")
results_csv_df

Successfully saved model results comparison to: ../outputs/model_results.csv


,model_name,accuracy,precision,recall,f1_score
0,Logistic Regression,0.6,0.6,1.0,0.75
1,Random Forest,0.6,0.6,1.0,0.75
